# Solutions — SQL and DataFrame API (Hard 12)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_franchises   = spark.table("samples.bakehouse.sales_franchises")
df_customers    = spark.table("samples.bakehouse.sales_customers")
df_orders       = spark.table("samples.tpch.orders")
df_customer     = spark.table("samples.tpch.customer")
df_bookings     = spark.table("samples.wanderbricks.bookings")

# Register temp views
df_transactions.createOrReplaceTempView("sales_transactions_view")
df_franchises.createOrReplaceTempView("franchises_view")
df_customers.createOrReplaceTempView("customers_view")
df_orders.createOrReplaceTempView("orders_view")
df_customer.createOrReplaceTempView("customer_view")
df_bookings.createOrReplaceTempView("bookings_view")


## Solution 1 — Register Views and Query with SQL

In [ ]:
result_1 = spark.sql("""
    SELECT   f.franchiseID,
             f.name         AS franchise_name,
             SUM(t.totalPrice)     AS total_revenue,
             COUNT(*)              AS transaction_count
    FROM     sales_transactions_view t
    JOIN     franchises_view         f  ON t.franchiseID = f.franchiseID
    GROUP BY f.franchiseID, f.name
    ORDER BY total_revenue DESC
""")


## Solution 2 — SQL CTEs for Multi-Step Logic

In [ ]:
result_2 = spark.sql("""
    WITH daily AS (
        SELECT CAST(dateTime AS DATE) AS sale_date,
               product,
               SUM(totalPrice) AS daily_revenue
        FROM   sales_transactions_view
        GROUP BY 1, 2
    ),
    ranked AS (
        SELECT sale_date, product, daily_revenue,
               RANK() OVER (PARTITION BY sale_date ORDER BY daily_revenue DESC) AS rnk
        FROM daily
    )
    SELECT sale_date, product, daily_revenue
    FROM   ranked
    WHERE  rnk = 1
    ORDER BY sale_date DESC
""")


## Solution 3 — SQL Window Functions

In [ ]:
result_3 = spark.sql("""
    SELECT franchiseID,
           SUM(totalPrice)                                   AS total_revenue,
           RANK() OVER (ORDER BY SUM(totalPrice) DESC)       AS rnk
    FROM   sales_transactions_view
    GROUP BY franchiseID
""")


## Solution 4 — DataFrame API vs SQL: Prove They're Equivalent

In [ ]:
# DataFrame API version
df_result = (
    df_transactions
    .groupBy("paymentMethod")
    .agg(F.count("*").alias("transaction_count"))
    .withColumn("pct_share",
        F.round(F.col("transaction_count") / F.sum("transaction_count").over(Window.orderBy(F.lit(1))).cast("double") * 100, 2)
    )
    .orderBy("paymentMethod")
)

# SQL version (equivalent)
sql_result = spark.sql("""
    SELECT paymentMethod,
           COUNT(*) AS transaction_count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_share
    FROM   sales_transactions_view
    GROUP BY paymentMethod
    ORDER BY paymentMethod
""")

result_4 = df_result   # both are equivalent; return the DataFrame API version


## Solution 5 — SQL Subquery Pattern (IN / EXISTS)

In [ ]:
# Orders with total price above the average — subquery pattern
result_5 = spark.sql("""
    SELECT o_orderkey, o_totalprice
    FROM   orders_view
    WHERE  o_totalprice > (SELECT AVG(o_totalprice) FROM orders_view)
    ORDER BY o_totalprice DESC
    LIMIT 100
""")


## Solution 6 — Explore the Catalog with spark.catalog

In [ ]:
tables = spark.catalog.listTables("samples.bakehouse")
result_6 = spark.createDataFrame(
    [(t.name,) for t in tables],
    ["tableName"]
)


## Solution 7 — DESCRIBE TABLE for Schema Metadata

In [ ]:
result_7 = spark.sql("DESCRIBE TABLE samples.bakehouse.sales_transactions")
# Returns col_name, data_type, comment — select first 5 rows as demo
result_7 = result_7.select("col_name", "data_type").limit(5)


## Solution 8 — Dynamic SQL Generation

In [ ]:
products = [r["product"] for r in df_transactions.select("product").distinct().collect()]

union_parts = " UNION ALL ".join(
    f"SELECT '{p}' AS product, COUNT(*) AS row_count FROM sales_transactions_view WHERE product = '{p}'"
    for p in products[:5]  # limit to first 5 for demo
)

result_8 = spark.sql(union_parts)
